# 任务 1：基于 ID3 算法的银行信贷数据分类

目标：  
	•理解 ID3 决策树算法 的原理。  
	•使用 NumPy 自己实现 决策树分类，不依赖 sklearn.tree.DecisionTreeClassifier。  
	•训练一个 信用评级模型，判断客户是否 值得信贷。  

任务描述：  
	1.加载银行信贷数据（可使用自定义数据或 UCI 数据集）。  
	2.编写 信息熵、信息增益 计算函数。  
	3.手动实现 ID3 算法，构建决策树。  
	4.进行 预测 和 准确率评估。  

In [32]:
import numpy as np
import pandas as pd

# 创建银行信贷数据集（示例）
data = {
    "年龄": ["青年", "青年", "中年", "老年", "老年", "老年", "中年", "青年", "青年", "老年", "青年", "中年", "中年", "老年"],
    "有工作": ["否", "否", "是", "是", "是", "否", "否", "否", "否", "是", "否", "是", "否", "是"],
    "有房子": ["否", "否", "是", "否", "是", "是", "是", "否", "是", "是", "是", "是", "否", "是"],
    "信贷情况": ["一般", "好", "好", "一般", "好", "好", "一般", "一般", "好", "非常好", "一般", "一般", "一般", "非常好"],
    "是否贷款": ["否", "否", "是", "是", "是", "否", "否", "否", "是", "是", "是", "是", "否", "是"]
}
df = pd.DataFrame(data)
df

,年龄,有工作,有房子,信贷情况,是否贷款
0,青年,否,否,一般,否
1,青年,否,否,好,否
2,中年,是,是,好,是
3,老年,是,否,一般,是
4,老年,是,是,好,是
5,老年,否,是,好,否
6,中年,否,是,一般,否
7,青年,否,否,一般,否
8,青年,否,是,好,是
9,老年,是,是,非常好,是


In [33]:
#计算信息熵
def entropy(y):
    values, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return -np.sum(probs * np.log2(probs))

In [5]:
y = [3/10,7/10]
HY = entropy(y)
print(HY)

1.0


In [7]:
# x1等于80
# x1left等于左边的信息熵
x1left = [2/2]
Hx1left = entropy(x1left)
print(Hx1left)

# x1right等于右边的信息熵
x1right = [3/8,5/8]
Hx1right = entropy(x1right)
print(Hx1right)

-0.0
1.0


In [34]:
print(df.columns)

Index(['年龄', '有工作', '有房子', '信贷情况', '是否贷款'], dtype='object')


In [35]:
#计算信息增益
def info_gain(df, feature, target):
    total_entropy = entropy(df[target])
    values, counts = np.unique(df[feature], return_counts=True)
    weighted_entropy = np.sum([(counts[i] / np.sum(counts)) * entropy(df.where(df[feature] == values[i]).dropna()[target]) for i in range(len(values))])
    return total_entropy - weighted_entropy

In [36]:
y = info_gain(df,'信贷情况','是否贷款')
print(y)

0.1458459985690298


In [27]:
data = [[60,'否'],[75,'否'],[85,'是'],[90,'是'],[95,'是'],[100,'是'],[100,'否'],[110,'否'],[125,'否'],[220,'否']]
df = pd.DataFrame(data,columns=['X','Y'])
feature = df.X
target = df.Y
print(target)
#info_gain(df,feature,target)

0    否
1    否
2    是
3    是
4    是
5    是
6    否
7    否
8    否
9    否
Name: Y, dtype: object


In [3]:
#递归构建ID3决策树
def id3(df, features, target):
    if len(np.unique(df[target])) == 1:
        return np.unique(df[target])[0]

    if len(features) == 0:
        return df[target].mode()[0]

    best_feature = max(features, key=lambda f: info_gain(df, f, target))
    tree = {best_feature: {}}
    
    for value in np.unique(df[best_feature]):
        sub_data = df.where(df[best_feature] == value).dropna()
        tree[best_feature][value] = id3(sub_data, [f for f in features if f != best_feature], target)

    return tree

In [6]:
#训练ID3决策树
features = ["年龄", "有工作", "有房子", "信贷情况"]
decision_tree = id3(df, features, "是否贷款")
print("ID3 生成的决策树:", decision_tree)

ID3 生成的决策树: {'有工作': {'否': {'有房子': {'否': '否', '是': {'年龄': {'中年': '否', '老年': '否', '青年': '是'}}}}, '是': '是'}}


# 任务 2：使用 DecisionTreeRegressor 预测波士顿房价

目标：  
	•了解 CART（回归决策树） 适用于回归问题。  
	•使用 sklearn.tree.DecisionTreeRegressor 预测 波士顿房价。  
	•评估模型的 均方误差（MSE） 和 R² 分数。  

任务描述：
	1.加载数据 sklearn.datasets.load_boston（波士顿房价数据）。  
	2.使用 train_test_split() 划分训练集和测试集。  
	3.训练 DecisionTreeRegressor 模型进行房价预测。  
	4.评估模型 的预测效果（均方误差、R²分数）。  
	5.可视化决策树 结构（可选）。  

In [8]:
from sklearn.datasets import load_boston
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [9]:
#加载波士顿房价数据
boston = load_boston()
X, y = boston.data, boston.target

/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function load_boston is deprecated; `load_boston` is deprecated in 1.0 and will be removed in 1.2.

    The Boston housing prices dataset has an ethical problem. You can refer to
    the documentation of this function for further details.

    The scikit-learn maintainers therefore strongly discourage the use of this
    dataset unless the purpose of the code is to study and educate about
    ethical issues in data science and machine learning.

    In this special case, you can fetch the dataset from the original
    source::

        import pandas as pd
        import numpy as np


        data_url = "http://lib.stat.cmu.edu/datasets/boston"
        raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
        data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
        target = raw_df.values[1::2, 2]

    Alternative datasets include the California housing dataset (

In [10]:
#拆分数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
#训练回归决策树
regressor = DecisionTreeRegressor(max_depth=5)
regressor.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=5)

In [12]:
#预测
y_pred = regressor.predict(X_test)

In [13]:
#评估
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"均方误差 (MSE): {mse:.2f}")
print(f"R² 分数: {r2:.2f}")

均方误差 (MSE): 8.55
R² 分数: 0.88


# 任务 3：数据集拆分

目标：  
	•熟练使用 train_test_split() 进行 数据集划分。  
	•掌握 训练集 vs. 测试集 的划分比例。  

任务描述：  
	1.选择一个数据集（可以是银行信贷数据或波士顿房价数据）。  
	2.使用 train_test_split() 按 80% 训练集，20% 测试集 进行划分。  
	3.打印数据集形状，确认拆分是否正确。  

In [14]:
from sklearn.model_selection import train_test_split

# 创建示例数据
data = {
    "年龄": ["青年", "青年", "中年", "老年", "老年", "老年", "中年", "青年", "青年", "老年", "青年", "中年", "中年", "老年"],
    "有工作": ["否", "否", "是", "是", "是", "否", "否", "否", "否", "是", "否", "是", "否", "是"],
    "是否贷款": ["否", "否", "是", "是", "是", "否", "否", "否", "是", "是", "是", "是", "否", "是"]
}

train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)
print(f"训练集大小: {train_data.shape}")
print(f"测试集大小: {test_data.shape}")

训练集大小: (11, 5)
测试集大小: (3, 5)
